# Guidance + Binding Affinity Training (Ligand-Context Readout)

This notebook implements a new guidance expert designed around the main problems in `guidance_wrapper_fixed.ipynb`:

- the pocket should be present during message passing, because ligand-pocket geometry matters;
- the final graph readout should use ligand nodes only, so the fixed pocket does not dominate the prediction;
- messages should include explicit distance information, because affinity-like targets are strongly distance-sensitive;
- edge semantics should be explicit, at minimum distinguishing ligand-ligand edges from pocket-to-ligand context edges.

This design is meant to stay compatible with TargetDiff guidance at sampling time. The runtime sampler only has:

- ligand coordinates,
- ligand atom-class predictions,
- pocket coordinates,
- pocket atomic numbers.

So this notebook avoids dependence on reconstructed bond graphs. Instead it uses only geometry and node-type-aware edge construction, which can be reproduced inside the sampler.

## Design Summary

Each training example is a combined ligand+pocket graph.

- Nodes:
  - ligand atoms
  - pocket atoms
- Node inputs:
  - atomic number `z`
  - node type (`ligand` or `pocket`)
- Edge types:
  - `ligand_ligand_spatial`
  - `pocket_to_ligand`
- Geometry:
  - a Gaussian radial basis embedding of pairwise distance on every edge
- Message passing:
  - ligand nodes receive messages from nearby ligand atoms and nearby pocket atoms
  - pocket nodes are treated as context and are not updated by default
- Readout:
  - pool only ligand node states
  - pass the pooled ligand representation through an MLP head to predict affinity

Why this is viable:

- the output is a scalar invariant to global translation and rotation, because it depends on pairwise distances;
- gradients with respect to ligand coordinates still exist through those distances, so the model can be used for sampling-time guidance;
- pocket context affects ligand states during message passing, but the fixed pocket no longer dominates the graph-level representation;
- the training graph and the runtime graph can be constructed from the same ingredients.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch.utils.data import Dataset, random_split
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.utils import scatter
from tqdm.auto import tqdm

import utils.transforms as trans

torch.manual_seed(0)

LIGAND_NODE = 1
POCKET_NODE = 0

EDGE_LIGAND_LIGAND = 0
EDGE_POCKET_TO_LIGAND = 1

PTABLE = Chem.GetPeriodicTable()

In [ ]:
def load_pocket(pdb_path):
    coords = []
    z_list = []

    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            elem = line[76:78].strip() or line[12:16].strip()[0]
            atomic_num = PTABLE.GetAtomicNumber(elem)
            coords.append([x, y, z])
            z_list.append(atomic_num)

    pocket_pos = torch.tensor(coords, dtype=torch.float32)
    pocket_z = torch.tensor(z_list, dtype=torch.long)
    return pocket_pos, pocket_z


def mol_to_ligand_tensors(mol, pred_pos):
    n_atoms = mol.GetNumAtoms()
    pos = torch.as_tensor(pred_pos, dtype=torch.float32)
    assert pos.shape[0] == n_atoms, f"pos has {pos.shape[0]} atoms, mol has {n_atoms}"
    z = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.long)
    return pos, z


def build_context_edges(pos, node_type, batch=None, r_ligand=5.0, r_cross=6.0):
    """
    Build a runtime-compatible directed context graph.

    Edge types:
    - ligand -> ligand spatial edges within r_ligand
    - pocket -> ligand context edges within r_cross

    Pocket nodes are included as message sources but not updated by default.
    """
    if batch is None:
        batch = torch.zeros(pos.size(0), dtype=torch.long, device=pos.device)

    edge_src = []
    edge_dst = []
    edge_type = []

    for b in batch.unique(sorted=True):
        mask_b = batch == b
        idx = mask_b.nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            continue

        sub_pos = pos[idx]
        sub_type = node_type[idx]

        lig_local = (sub_type == LIGAND_NODE).nonzero(as_tuple=True)[0]
        poc_local = (sub_type == POCKET_NODE).nonzero(as_tuple=True)[0]

        if lig_local.numel() > 0:
            lig_pos = sub_pos[lig_local]
            dist_ll = torch.cdist(lig_pos, lig_pos, p=2)
            keep_ll = dist_ll <= r_ligand
            keep_ll = keep_ll & ~torch.eye(lig_local.numel(), dtype=torch.bool, device=pos.device)
            src_ll, dst_ll = keep_ll.nonzero(as_tuple=True)
            if src_ll.numel() > 0:
                edge_src.append(idx[lig_local[src_ll]])
                edge_dst.append(idx[lig_local[dst_ll]])
                edge_type.append(torch.full((src_ll.numel(),), EDGE_LIGAND_LIGAND, dtype=torch.long, device=pos.device))

        if lig_local.numel() > 0 and poc_local.numel() > 0:
            lig_pos = sub_pos[lig_local]
            poc_pos = sub_pos[poc_local]
            dist_pl = torch.cdist(poc_pos, lig_pos, p=2)
            keep_pl = dist_pl <= r_cross
            src_pl, dst_pl = keep_pl.nonzero(as_tuple=True)
            if src_pl.numel() > 0:
                edge_src.append(idx[poc_local[src_pl]])
                edge_dst.append(idx[lig_local[dst_pl]])
                edge_type.append(torch.full((src_pl.numel(),), EDGE_POCKET_TO_LIGAND, dtype=torch.long, device=pos.device))

    if len(edge_src) == 0:
        return (
            torch.empty(2, 0, dtype=torch.long, device=pos.device),
            torch.empty(0, dtype=torch.long, device=pos.device),
        )

    edge_index = torch.stack([torch.cat(edge_src), torch.cat(edge_dst)], dim=0)
    edge_type = torch.cat(edge_type, dim=0)
    return edge_index, edge_type

## Dataset

The dataset intentionally mirrors the information that will be available inside TargetDiff guidance.

Each graph stores:

- `pos`: ligand positions concatenated with pocket positions
- `z`: ligand atomic numbers concatenated with pocket atomic numbers
- `node_type`: ligand vs pocket
- `edge_index`, `edge_type`: spatial ligand-ligand and pocket-to-ligand edges
- `y`: scalar affinity label

No bond graph is required. That keeps training and runtime aligned.

In [ ]:
class LigandContextFromPT(Dataset):
    def __init__(
        self,
        pt_path=None,
        pt_dir=None,
        pdb_path=None,
        vina_mode="score_only",
        max_affinity=0.0,
        results_key="all_results",
        r_ligand=5.0,
        r_cross=6.0,
    ):
        assert pdb_path is not None, "pdb_path is required"
        assert (pt_path is not None) or (pt_dir is not None), "Provide pt_path or pt_dir"
        assert vina_mode in ["dock", "minimize", "score_only"]

        self.vina_mode = vina_mode
        self.max_affinity = max_affinity
        self.r_ligand = r_ligand
        self.r_cross = r_cross

        results = []
        pt_files = []
        if pt_dir is not None:
            for p in sorted(Path(pt_dir).glob("*.pt")):
                obj = torch.load(p, map_location="cpu", weights_only=False)
                if results_key in obj:
                    results.extend(obj[results_key])
                elif "results" in obj:
                    results.extend(obj["results"])
                pt_files.append(p)
        else:
            obj = torch.load(pt_path, map_location="cpu", weights_only=False)
            if results_key in obj:
                results.extend(obj[results_key])
            elif "results" in obj:
                results.extend(obj["results"])
            pt_files.append(pt_path)

        self.results = results
        self.pt_files = pt_files
        self.pocket_pos, self.pocket_z = load_pocket(pdb_path)

        if max_affinity is None:
            self.indices = list(range(len(self.results)))
        else:
            self.indices = [
                i for i, r in enumerate(self.results)
                if r["vina"][self.vina_mode][0]["affinity"] <= max_affinity
            ]
            if len(self.indices) == 0:
                raise ValueError("All samples filtered out; relax max_affinity")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        rec = self.results[self.indices[idx]]
        mol = rec["mol"]
        pred_pos = rec["pred_pos"]
        vina_rec = rec["vina"][self.vina_mode][0]

        lig_pos, z_lig = mol_to_ligand_tensors(mol, pred_pos)
        pos = torch.cat([lig_pos, self.pocket_pos], dim=0)
        z = torch.cat([z_lig, self.pocket_z], dim=0)
        node_type = torch.cat([
            torch.ones(lig_pos.size(0), dtype=torch.long),
            torch.zeros(self.pocket_pos.size(0), dtype=torch.long),
        ])

        edge_index, edge_type = build_context_edges(
            pos=pos,
            node_type=node_type,
            batch=None,
            r_ligand=self.r_ligand,
            r_cross=self.r_cross,
        )

        y = torch.tensor([vina_rec["affinity"]], dtype=torch.float32)
        data = Data(
            pos=pos,
            z=z,
            node_type=node_type,
            edge_index=edge_index,
            edge_type=edge_type,
            y=y,
            smiles=rec.get("smiles"),
        )

        chem = rec.get("chem_results", {})
        for key in ("qed", "sa", "logp"):
            if key in chem:
                data[key] = torch.tensor([chem[key]], dtype=torch.float32)

        return data

## Model

This is an invariant geometric message-passing model.

- It is not a plain GAT.
- It does not require full e3nn-style tensor features.
- It uses only quantities that remain available during sampling: atom types, node types, and pairwise distances.

Important behavior:

- ligand nodes are updated by messages from nearby ligand atoms and nearby pocket atoms;
- pocket nodes are kept fixed as context by default;
- the final binding prediction is made from pooled ligand states only.

That makes the model much closer to a contextualized ligand energy model than an all-node graph regressor dominated by the fixed pocket.

In [ ]:
class GaussianRBF(nn.Module):
    def __init__(self, num_rbf=32, cutoff=6.0):
        super().__init__()
        centers = torch.linspace(0.0, cutoff, num_rbf)
        self.register_buffer("centers", centers)
        self.gamma = 1.0 / max((centers[1] - centers[0]).item() ** 2, 1e-6) if num_rbf > 1 else 1.0

    def forward(self, dist):
        diff = dist.unsqueeze(-1) - self.centers.unsqueeze(0)
        return torch.exp(-self.gamma * diff.pow(2))


class ContextMessageBlock(nn.Module):
    def __init__(self, hidden, num_edge_types=2, num_rbf=32, cutoff=6.0, dropout=0.1):
        super().__init__()
        self.rbf = GaussianRBF(num_rbf=num_rbf, cutoff=cutoff)
        self.edge_type_emb = nn.Embedding(num_edge_types, hidden)
        self.msg_mlp = nn.Sequential(
            nn.Linear(2 * hidden + hidden + num_rbf, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
        )
        self.upd_mlp = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
        )
        self.norm = nn.LayerNorm(hidden)

    def forward(self, h, pos, edge_index, edge_type, node_type):
        if edge_index.numel() == 0:
            return h

        src, dst = edge_index
        rel = pos[src] - pos[dst]
        dist = rel.norm(dim=-1)
        radial = self.rbf(dist)
        edge_kind = self.edge_type_emb(edge_type)

        m_in = torch.cat([h[src], h[dst], edge_kind, radial], dim=-1)
        m_ij = self.msg_mlp(m_in)
        m_i = scatter(m_ij, dst, dim=0, dim_size=h.size(0), reduce="mean")

        upd = self.upd_mlp(torch.cat([h, m_i], dim=-1))
        lig_mask = node_type == LIGAND_NODE
        out = h.clone()
        out[lig_mask] = self.norm(h[lig_mask] + upd[lig_mask])
        return out


class LigandContextAffinityModel(nn.Module):
    def __init__(self, max_z=100, hidden=128, n_layers=4, num_rbf=32, cutoff=6.0, dropout=0.1):
        super().__init__()
        self.z_emb = nn.Embedding(max_z, hidden)
        self.node_type_emb = nn.Embedding(2, hidden)
        self.input_norm = nn.LayerNorm(hidden)
        self.layers = nn.ModuleList([
            ContextMessageBlock(hidden=hidden, num_edge_types=2, num_rbf=num_rbf, cutoff=cutoff, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, data):
        pos = data.pos
        z = data.z
        node_type = data.node_type.long()
        edge_index = data.edge_index
        edge_type = data.edge_type.long()

        batch = getattr(data, "batch", None)
        if batch is None:
            batch = torch.zeros(pos.size(0), dtype=torch.long, device=pos.device)

        h = self.z_emb(z) + self.node_type_emb(node_type)
        h = self.input_norm(h)

        for layer in self.layers:
            h = layer(h, pos, edge_index, edge_type, node_type)

        lig_mask = (node_type == LIGAND_NODE).float()
        lig_counts = scatter(lig_mask, batch, dim=0, reduce="sum").clamp(min=1.0)
        g = scatter(h * lig_mask.unsqueeze(-1), batch, dim=0, reduce="sum")
        g = g / lig_counts.unsqueeze(-1)
        affinity = self.head(g).squeeze(-1)
        return affinity


class GuidedLigandContextWrapper(nn.Module):
    def __init__(
        self,
        affinity_model,
        pocket_pos,
        pocket_z,
        ligand_atom_mode="add_aromatic",
        r_ligand=5.0,
        r_cross=6.0,
        device="cpu",
    ):
        super().__init__()
        self.affinity_model = affinity_model
        self.ligand_atom_mode = ligand_atom_mode
        self.r_ligand = r_ligand
        self.r_cross = r_cross

        pocket_pos = pocket_pos.to(device)
        pocket_center = pocket_pos.mean(dim=0, keepdim=True)
        self.register_buffer("pocket_pos_centered", pocket_pos - pocket_center)
        self.register_buffer("pocket_z", pocket_z.to(device).long())
        self.num_pocket_atoms = pocket_pos.size(0)

    def forward(self, ligand_pos, ligand_v, batch_ligand, batch_protein=None, protein_pos=None):
        device = ligand_pos.device
        num_graphs = int(batch_ligand.max().item()) + 1

        atomic_numbers = trans.get_atomic_number_from_index(
            ligand_v.detach().cpu(),
            mode=self.ligand_atom_mode,
        )
        z_lig = torch.tensor(atomic_numbers, dtype=torch.long, device=device)

        use_provided_protein = (
            protein_pos is not None
            and batch_protein is not None
            and protein_pos.size(0) == self.num_pocket_atoms * num_graphs
        )

        if use_provided_protein:
            pocket_pos = protein_pos
            pocket_batch = batch_protein
        else:
            pocket_pos = self.pocket_pos_centered.repeat(num_graphs, 1)
            pocket_batch = torch.arange(num_graphs, device=device).repeat_interleave(self.num_pocket_atoms)

        z_pocket = self.pocket_z.repeat(num_graphs).to(device)

        pos = torch.cat([ligand_pos, pocket_pos], dim=0)
        z = torch.cat([z_lig, z_pocket], dim=0)
        batch = torch.cat([batch_ligand, pocket_batch], dim=0)
        node_type = torch.cat([
            torch.ones(ligand_pos.size(0), dtype=torch.long, device=device),
            torch.zeros(pocket_pos.size(0), dtype=torch.long, device=device),
        ])

        edge_index, edge_type = build_context_edges(
            pos=pos,
            node_type=node_type,
            batch=batch,
            r_ligand=self.r_ligand,
            r_cross=self.r_cross,
        )

        data = Data(
            pos=pos,
            z=z,
            node_type=node_type,
            batch=batch,
            edge_index=edge_index,
            edge_type=edge_type,
        )
        affinity = self.affinity_model(data)
        return -affinity

## Training Configuration

Recommended defaults for a first `score_only` run:

- `VINA_MODE = "score_only"`
- `MAX_AFFINITY = 0.0` to remove obvious positive-score clash outliers
- Huber loss instead of plain MSE
- a mode-specific checkpoint filename so label switches do not silently resume the wrong model

In [ ]:
PT_DIR = "metrics_dir"
PT_PATH = None
PDB_PATH = "y220c_pocket10.pdb"

VINA_MODE = "score_only"
MAX_AFFINITY = 0.0

R_LIGAND = 5.0
R_CROSS = 6.0

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
BATCH_SIZE = 32

HIDDEN = 128
N_LAYERS = 4
NUM_RBF = 32
DROPOUT = 0.1

LR = 3e-4
NUM_EPOCHS = 20

pt_kwargs = {"pt_dir": PT_DIR, "pt_path": PT_PATH}
pt_kwargs = {k: v for k, v in pt_kwargs.items() if v is not None}

ds = LigandContextFromPT(
    pdb_path=PDB_PATH,
    vina_mode=VINA_MODE,
    max_affinity=MAX_AFFINITY,
    r_ligand=R_LIGAND,
    r_cross=R_CROSS,
    **pt_kwargs,
)

print(f"Loaded {len(ds)} graphs")
if len(ds) == 0:
    raise ValueError("No samples loaded; check PT_DIR/PT_PATH")

n_total = len(ds)
n_train = int(TRAIN_RATIO * n_total)
remaining = n_total - n_train
n_val = int(VAL_RATIO * n_total)
if n_val > remaining:
    n_val = remaining
n_test = max(0, n_total - n_train - n_val)

train_dataset, val_dataset, test_dataset = random_split(
    ds,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(0),
)

train_loader = PyGDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = PyGDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = PyGDataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Splits -> train {len(train_dataset)}, val {len(val_dataset)}, test {len(test_dataset)}")
print(ds[0])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LigandContextAffinityModel(
    max_z=100,
    hidden=HIDDEN,
    n_layers=N_LAYERS,
    num_rbf=NUM_RBF,
    cutoff=max(R_LIGAND, R_CROSS),
    dropout=DROPOUT,
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

ckpt_dir = Path("checkpoints")
ckpt_dir.mkdir(exist_ok=True)
ckpt_path = ckpt_dir / f"ligand_context_{VINA_MODE}.pt"
start_epoch = 0

if ckpt_path.exists():
    obj = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(obj["model"])
    opt.load_state_dict(obj["optimizer"])
    if obj.get("scheduler") is not None:
        scheduler.load_state_dict(obj["scheduler"])
    start_epoch = obj.get("epoch", 0) + 1
    print(f"[ckpt] resumed from epoch {start_epoch - 1}")


def eval_loader(loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            pred = model(data)
            true = data.y.to(pred.dtype).view(-1)
            loss = F.smooth_l1_loss(pred, true)
            total_loss += loss.item() * data.num_graphs
    return total_loss / max(1, len(loader.dataset))


for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for data in tqdm(train_loader, desc=f"epoch {epoch}"):
        data = data.to(device)
        pred = model(data)
        true = data.y.to(pred.dtype).view(-1)
        loss = F.smooth_l1_loss(pred, true)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * data.num_graphs

    scheduler.step()
    train_loss = total_loss / max(1, len(train_loader.dataset))
    val_loss = eval_loader(val_loader) if len(val_loader.dataset) > 0 else float("nan")
    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": scheduler.state_dict(),
        },
        ckpt_path,
    )

In [ ]:
def collect_preds(model, loader, device):
    model.eval()
    ys = []
    ys_pred = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            pred = model(data)
            true = data.y.to(pred.dtype).view(-1)
            ys.append(true.cpu())
            ys_pred.append(pred.cpu())
    if len(ys) == 0:
        return None, None
    return torch.cat(ys, dim=0), torch.cat(ys_pred, dim=0)


ys, ys_pred = collect_preds(model, test_loader, device)
if ys is not None:
    plt.figure(figsize=(5, 5))
    plt.scatter(ys, ys_pred, s=5, alpha=0.5)
    min_v = min(ys.min().item(), ys_pred.min().item())
    max_v = max(ys.max().item(), ys_pred.max().item())
    plt.plot([min_v, max_v], [min_v, max_v], "r--")
    plt.xlabel("True affinity")
    plt.ylabel("Predicted affinity")
    plt.title("Parity plot (test set)")
    plt.tight_layout()
    plt.show()
    mae = torch.mean(torch.abs(ys_pred - ys)).item()
    print("Test MAE:", f"{mae:.4f}")
else:
    print("No test samples to plot")